# Contextual-Top2Vec Reproduction: Preliminary Feasibility Assessment

This notebook conducts a preliminary reproducibility assessment of Contextual-Top2Vec (C-Top2Vec), introduced by Angelov and Inkpen (2024). The initial objective is to verify whether the published implementation can be successfully executed using the 20 Newsgroups dataset within the available local computational environment.

The feasibility stage focuses on reproducing the principal components of the C-Top2Vec pipeline, including contextual embedding generation, dimensionality reduction, density-based topic discovery, document-topic distributions, topic relevance scores, and token-level topic assignments. A reduced stratified sample is initially utilized to confirm implementation feasibility prior to conducting the larger-scale reproduction.

In [1]:
import sys
import numpy as np
import pandas as pd
import torch

from top2vec import Top2Vec

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("Top2Vec import: SUCCESS")

/usr/local/Caskroom/miniforge/base/envs/ctop2vec-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.15
NumPy: 1.26.4
PyTorch: 2.2.2
MPS available: True
Top2Vec import: SUCCESS


## 1. Load the 20 Newsgroups Dataset

The 20 Newsgroups dataset is utilized as the initial corpus for the feasibility assessment as it is one of the two primary datasets used by Angelov and Inkpen (2024). The corpus consists of 18,846 documents distributed across 20 predefined newsgroup categories, providing ground-truth topic labels that can subsequently support clustering-based evaluation.

The complete textual content is retained during retrieval. This allows the dataset to be processed according to the filtering criteria explicitly reported in the original study, rather than applying additional text-removal procedures that may alter the experimental corpus.

In [2]:
from sklearn.datasets import fetch_20newsgroups

dataset = fetch_20newsgroups(
    subset="all",
    remove=()
)

print("Total documents:", len(dataset.data))
print("Number of categories:", len(dataset.target_names))

print("\nCategories:")
for category in dataset.target_names:
    print("-", category)

Total documents: 18846
Number of categories: 20

Categories:
- alt.atheism
- comp.graphics
- comp.os.ms-windows.misc
- comp.sys.ibm.pc.hardware
- comp.sys.mac.hardware
- comp.windows.x
- misc.forsale
- rec.autos
- rec.motorcycles
- rec.sport.baseball
- rec.sport.hockey
- sci.crypt
- sci.electronics
- sci.med
- sci.space
- soc.religion.christian
- talk.politics.guns
- talk.politics.mideast
- talk.politics.misc
- talk.religion.misc


## 2. Dataset Verification

Prior to model implementation, the retrieved corpus is converted into a structured dataframe to verify its size, category distribution, and textual completeness. The original category labels are retained as reference information but are not supplied to Contextual-Top2Vec during topic discovery.

This distinction is necessary because C-Top2Vec operates as an unsupervised topic modelling approach. The known labels are therefore reserved exclusively for subsequent evaluation, including Adjusted Mutual Information (AMI) and Adjusted Rand Index (ARI).

In [5]:
df = pd.DataFrame({
    "text": dataset.data,
    "target": dataset.target
})

df["category"] = df["target"].map(
    lambda x: dataset.target_names[x]
)

print("Dataset shape:", df.shape)
print("Number of categories:", df["category"].nunique())
print("Missing documents:", df["text"].isna().sum())
print("Empty documents:", (df["text"].str.strip() == "").sum())

print("\nDocuments per category:")
print(df["category"].value_counts().sort_index())

Dataset shape: (18846, 3)
Number of categories: 20
Missing documents: 0
Empty documents: 0

Documents per category:
category
alt.atheism                 799
comp.graphics               973
comp.os.ms-windows.misc     985
comp.sys.ibm.pc.hardware    982
comp.sys.mac.hardware       963
comp.windows.x              988
misc.forsale                975
rec.autos                   990
rec.motorcycles             996
rec.sport.baseball          994
rec.sport.hockey            999
sci.crypt                   991
sci.electronics             984
sci.med                     990
sci.space                   987
soc.religion.christian      997
talk.politics.guns          910
talk.politics.mideast       940
talk.politics.misc          775
talk.religion.misc          628
Name: count, dtype: int64


## 3. Dataset Preprocessing

Following the procedure reported by Angelov and Inkpen (2024), empty documents and documents containing fewer than 35 characters are removed prior to model implementation. No stemming, lemmatization, or stop-word removal is applied at this stage, as the contextual model relies on the semantic and syntactic information contained within the original text.

The preprocessing procedure is deliberately limited to the criteria explicitly reported in the original study to maintain consistency with the intended reproduction.

In [6]:
original_size = len(df)

df_clean = df[
    df["text"].notna() &
    (df["text"].str.strip() != "")
].copy()

df_clean = df_clean[
    df_clean["text"].str.len() >= 35
].copy()

df_clean.reset_index(drop=True, inplace=True)

print("Original documents:", original_size)
print("Documents after filtering:", len(df_clean))
print("Documents removed:", original_size - len(df_clean))

print("\nMinimum document length:", df_clean["text"].str.len().min())
print("Maximum document length:", df_clean["text"].str.len().max())
print("Mean document length:",
      round(df_clean["text"].str.len().mean(), 2))

Original documents: 18846
Documents after filtering: 18846
Documents removed: 0

Minimum document length: 115
Maximum document length: 160616
Mean document length: 1902.53


## 4. Stratified Feasibility Sample

An initial attempt to execute Contextual-Top2Vec using the complete 20 Newsgroups corpus exceeded the available Apple MPS memory during contextual embedding generation. Accordingly, a reduced stratified sample of 200 documents is utilized for the preliminary feasibility assessment.

Stratified sampling retains representation across all 20 original categories while substantially reducing the computational requirements. The category labels are used solely for constructing the sample and are not provided to the topic model during training.

In [7]:
from sklearn.model_selection import train_test_split

sample_df, _ = train_test_split(
    df_clean,
    train_size=200,
    stratify=df_clean["target"],
    random_state=42
)

sample_df = sample_df.reset_index(drop=True)

sample_documents = sample_df["text"].tolist()
sample_labels = sample_df["target"].to_numpy()

print("Sample documents:", len(sample_documents))
print("Categories represented:", sample_df["target"].nunique())

print("\nDocuments per category:")
print(sample_df["category"].value_counts().sort_index())

Sample documents: 200
Categories represented: 20

Documents per category:
category
alt.atheism                  8
comp.graphics               10
comp.os.ms-windows.misc     10
comp.sys.ibm.pc.hardware    10
comp.sys.mac.hardware       10
comp.windows.x              10
misc.forsale                10
rec.autos                   11
rec.motorcycles             11
rec.sport.baseball          11
rec.sport.hockey            11
sci.crypt                   11
sci.electronics             10
sci.med                     11
sci.space                   10
soc.religion.christian      11
talk.politics.guns          10
talk.politics.mideast       10
talk.politics.misc           8
talk.religion.misc           7
Name: count, dtype: int64


## 5. Contextual-Top2Vec Feasibility Configuration

The feasibility model retains the central contextual components of the method, including the `all-mpnet-base-v2` embedding model and a contextual smoothing window of three. Given the substantially reduced corpus size, the density-based clustering thresholds are lowered relative to those reported for the complete dataset.

Accordingly, the feasibility configuration uses 15 UMAP neighbours, an HDBSCAN minimum cluster size of five, and a minimum vocabulary count of five. These modifications are intended only to determine whether the complete C-Top2Vec pipeline can be executed locally and should not be interpreted as a direct reproduction of the quantitative results reported by Angelov and Inkpen (2024).

In [8]:
import time

umap_args = {
    "n_neighbors": 15,
    "n_components": 5
}

hdbscan_args = {
    "min_cluster_size": 5
}

print("Starting preliminary Contextual-Top2Vec feasibility run...")
print("Documents:", len(sample_documents))

start_time = time.time()

model = Top2Vec(
    documents=sample_documents,
    contextual_top2vec=True,
    c_top2vec_smoothing_window=3,
    embedding_model="all-mpnet-base-v2",
    min_count=5,
    umap_args=umap_args,
    hdbscan_args=hdbscan_args,
    workers=4,
    verbose=True
)

elapsed = time.time() - start_time

print("\nTraining completed.")
print(f"Runtime: {elapsed / 60:.2f} minutes")
print("Topics discovered:", model.get_num_topics())

2026-08-22 00:30:14,258 - top2vec - INFO - Pre-processing documents for training
/usr/local/Caskroom/miniforge/base/envs/ctop2vec-env/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-08-22 00:30:14,389 - top2vec - INFO - Creating vocabulary embedding


Starting preliminary Contextual-Top2Vec feasibility run...
Documents: 200


Embedding vocabulary:  26%|████████████████▏                                            | 9/34 [00:38<01:47,  4.32s/it]


RuntimeError: MPS backend out of memory (MPS allocated: 1.63 GB, other allocations: 4.80 GB, max allowed: 6.77 GB). Tried to allocate 384.00 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).